# ✅ Gabarito — Módulo 4: Análise Exploratória de Dados

**Curso Introdutório de Python para Ciência de Dados — UNIFOR T326**

> 🔒 **Este arquivo contém as soluções completas.** Tente resolver os exercícios antes de consultar!

---

In [ ]:
# Setup — idêntico ao arquivo de exercícios
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.dpi': 110, 'figure.facecolor': 'white',
                     'axes.spines.top': False, 'axes.spines.right': False})

COR_REGIAO = {'Norte':'#1f77b4','Nordeste':'#ff7f0e','Centro-Oeste':'#2ca02c',
              'Sudeste':'#d62728','Sul':'#9467bd'}

np.random.seed(7)
regioes_info = {
    'Norte':        {'estados': ['AM','PA','AC','RO','RR','AP','TO'], 'n': 620,  'idhm': (0.615, 0.05)},
    'Nordeste':     {'estados': ['MA','PI','CE','RN','PB','PE','AL','SE','BA'], 'n': 1950, 'idhm': (0.598, 0.055)},
    'Centro-Oeste': {'estados': ['MT','MS','GO','DF'], 'n': 550,  'idhm': (0.703, 0.044)},
    'Sudeste':      {'estados': ['MG','ES','RJ','SP'], 'n': 1240, 'idhm': (0.735, 0.05)},
    'Sul':          {'estados': ['PR','SC','RS'],      'n': 1210, 'idhm': (0.748, 0.038)},
}
rows = []
for reg, info in regioes_info.items():
    mu, sigma = info['idhm']
    for _ in range(info['n']):
        idhm = np.clip(np.random.normal(mu, sigma), 0.35, 0.95)
        pib_pc = max(2500, idhm * 58000 + np.random.normal(0, 9000))
        pop = int(np.clip(np.random.lognormal(9, 1.5), 800, 12_000_000))
        area = round(np.random.lognormal(7, 1.2), 1)
        rows.append({
            'CITY': f'Municipio_{len(rows):04d}',
            'STATE': np.random.choice(info['estados']),
            'REGION': reg,
            'IDHM': round(idhm, 3),
            'IDHM_Renda': round(np.clip(idhm + np.random.normal(0, 0.03), 0.3, 0.95), 3),
            'IDHM_Longevidade': round(np.clip(idhm + np.random.normal(0.025, 0.02), 0.4, 0.95), 3),
            'IDHM_Educacao': round(np.clip(idhm - np.random.normal(0.025, 0.03), 0.25, 0.90), 3),
            'GDP_CAPITA': round(pib_pc, 2),
            'IBGE_POP': pop,
            'AREA': area,
        })
df = pd.DataFrame(rows)
df['DENSIDADE'] = (df['IBGE_POP'] / df['AREA']).round(2)
df['GDP'] = (df['GDP_CAPITA'] * df['IBGE_POP']).round(0).astype(int)
idx_nulos = np.random.choice(df.index, size=80, replace=False)
df.loc[idx_nulos[:40], 'GDP_CAPITA'] = np.nan
df.loc[idx_nulos[40:], 'IDHM_Educacao'] = np.nan

print(f'✅ Dataset: {df.shape[0]} municípios, {df.shape[1]} colunas')

---
## ✅ Gabarito — Exercício 1: Inspeção Inicial

In [ ]:
# 1. Primeiras e últimas linhas
print('=== PRIMEIRAS 5 LINHAS ===')
display(df.head())
print('\n=== ÚLTIMAS 5 LINHAS ===')
display(df.tail())

# 2. Dimensões
print(f'\n=== DIMENSÕES ===')
print(f'  Linhas  : {df.shape[0]:,}')
print(f'  Colunas : {df.shape[1]}')

# 3. Valores nulos
print('\n=== VALORES NULOS ===')
nulos = df.isnull().sum()
pct   = (df.isnull().mean() * 100).round(2)
resumo = pd.DataFrame({'Nulos': nulos, '% do Total': pct})
print(resumo[resumo['Nulos'] > 0])

# 4. Estatísticas descritivas selecionadas
print('\n=== ESTATÍSTICAS DESCRITIVAS ===')
display(df[['IDHM', 'GDP_CAPITA', 'IBGE_POP']].describe().round(3))

---
## ✅ Gabarito — Exercício 2: Histograma

In [ ]:
fig, ax = plt.subplots(figsize=(9, 5))

# Histograma
ax.hist(df['IDHM'].dropna(), bins=35, color='steelblue', edgecolor='white', alpha=0.85)

# Média e mediana
media   = df['IDHM'].mean()
mediana = df['IDHM'].median()
ax.axvline(media,   color='red',    linestyle='--', linewidth=2, label=f'Média: {media:.3f}')
ax.axvline(mediana, color='orange', linestyle='-',  linewidth=2, label=f'Mediana: {mediana:.3f}')

# Formatação
ax.set_title('Distribuição do IDHM nos Municípios Brasileiros', fontsize=12, fontweight='bold')
ax.set_xlabel('IDHM')
ax.set_ylabel('Número de Municípios')
ax.legend()

plt.tight_layout()
plt.show()

# Estatísticas
print(f'Média          : {media:.4f}')
print(f'Mediana        : {mediana:.4f}')
print(f'Desvio-padrão  : {df["IDHM"].std():.4f}')

---
## ✅ Gabarito — Exercício 3: Agrupamento e Boxplot

In [ ]:
# Tabela de estatísticas por região
stats_regiao = (
    df.groupby('REGION')['IDHM']
    .agg(Media='mean', Mediana='median', DesvPad='std',
         Minimo='min', Maximo='max', N_Municipios='count')
    .round(3)
    .sort_values('Mediana', ascending=False)
)
print('=== IDHM POR REGIÃO ===')
display(stats_regiao)

# Boxplot por região (ordem decrescente de mediana)
ordem = stats_regiao.index.tolist()
fig, ax = plt.subplots(figsize=(10, 5))

dados_bp = [df[df['REGION'] == r]['IDHM'].dropna().values for r in ordem]
bp = ax.boxplot(dados_bp, labels=ordem, patch_artist=True,
                medianprops=dict(color='black', linewidth=2),
                flierprops=dict(marker='o', markersize=3, alpha=0.3))

for patch, regiao in zip(bp['boxes'], ordem):
    patch.set_facecolor(COR_REGIAO[regiao])
    patch.set_alpha(0.7)

ax.set_title('Distribuição do IDHM por Região\n(ordenado por mediana decrescente)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('IDHM')
ax.set_xlabel('Região')
plt.tight_layout()
plt.show()

---
## ✅ Gabarito — Exercício 4: Correlação

In [ ]:
cols = ['IDHM', 'IDHM_Renda', 'IDHM_Longevidade', 'IDHM_Educacao', 'GDP_CAPITA', 'DENSIDADE']
corr = df[cols].corr().round(3)

# Heatmap
fig, ax = plt.subplots(figsize=(8, 6))
mascara = np.triu(np.ones_like(corr, dtype=bool), k=1)  # Oculta triângulo superior
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn', center=0,
            mask=mascara, ax=ax, linewidths=0.5, vmin=-1, vmax=1, square=True)
ax.set_title('Matriz de Correlação', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

# Variável com maior correlação com IDHM (excluindo os 3 componentes)
excluir = ['IDHM', 'IDHM_Renda', 'IDHM_Longevidade', 'IDHM_Educacao']
corr_idhm = corr['IDHM'].drop(labels=excluir).abs().sort_values(ascending=False)
print(f'\n📊 Correlações com IDHM (excluindo componentes):')
print(corr_idhm)
print(f'\n🏆 Maior correlação com IDHM: {corr_idhm.idxmax()} ({corr_idhm.max():.3f})')

---
## ✅ Gabarito — Exercício 5: Scatter Plot

In [ ]:
# Remover outliers de GDP_CAPITA (acima do percentil 95)
p95 = df['GDP_CAPITA'].quantile(0.95)
df_plot = df[df['GDP_CAPITA'] <= p95].dropna(subset=['GDP_CAPITA', 'IDHM'])

fig, ax = plt.subplots(figsize=(11, 6))

# Pontos coloridos por região
for regiao, grupo in df_plot.groupby('REGION'):
    ax.scatter(grupo['GDP_CAPITA'], grupo['IDHM'],
               color=COR_REGIAO[regiao], label=regiao,
               alpha=0.4, s=15, edgecolors='none')

# Linha de tendência
coefs = np.polyfit(df_plot['GDP_CAPITA'], df_plot['IDHM'], 1)
x_range = np.linspace(df_plot['GDP_CAPITA'].min(), df_plot['GDP_CAPITA'].max(), 200)
ax.plot(x_range, np.polyval(coefs, x_range), 'k--', linewidth=2, label='Tendência')

# Formatação
ax.set_title('PIB per Capita × IDHM por Município e Região', fontsize=12, fontweight='bold')
ax.set_xlabel('PIB per Capita (R$)')
ax.set_ylabel('IDHM')
ax.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'R${x/1000:.0f}k'))
ax.legend(title='Região', framealpha=0.8)
plt.tight_layout()
plt.show()

print('\n📖 Interpretação:')
print('  Há uma relação positiva clara entre PIB per capita e IDHM: municípios mais ricos')
print('  tendem a ter maior desenvolvimento humano. Entretanto, a dispersão é alta,')
print('  especialmente para municípios com PIB baixo-médio — riqueza não garante desenvolvimento.')
print('  Municípios do Sul/Sudeste dominam os quadrantes de alto PIB e alto IDHM.')

---
## ✅ Gabarito — Exercício 6: Análise por Faixas de Porte

In [ ]:
# 1. Criando coluna PORTE
bins_pop  = [0, 5000, 20000, 100000, 500000, float('inf')]
labels_pop = ['Muito Pequeno\n(<5k)', 'Pequeno\n(5k–20k)',
               'Médio\n(20k–100k)', 'Grande\n(100k–500k)', 'Metrópole\n(≥500k)']
df['PORTE'] = pd.cut(df['IBGE_POP'], bins=bins_pop, labels=labels_pop)

# 2. IDHM por porte
idhm_porte = (
    df.groupby('PORTE', observed=True)['IDHM']
    .agg(IDHM_Medio='mean', Desvio='std', N_Municipios='count')
    .reset_index()
    .round(3)
)
print('=== IDHM POR PORTE ===')
print(idhm_porte.to_string(index=False))

# 3. Gráfico de barras com erro
fig, ax = plt.subplots(figsize=(10, 5))
cores = sns.color_palette('Blues', n_colors=len(idhm_porte))
barras = ax.bar(idhm_porte['PORTE'], idhm_porte['IDHM_Medio'],
                color=cores, edgecolor='white',
                yerr=idhm_porte['Desvio'], capsize=5,
                error_kw={'ecolor': 'gray', 'linewidth': 1.5})

for b, row in zip(barras, idhm_porte.itertuples()):
    ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.015,
            f'{row.IDHM_Medio:.3f}\n(n={row.N_Municipios:,})',
            ha='center', fontsize=8)

ax.set_title('IDHM Médio por Porte do Município\n(barras de erro = desvio-padrão)',
             fontsize=12, fontweight='bold')
ax.set_ylabel('IDHM Médio')
ax.set_ylim(0.5, 0.9)
plt.tight_layout()
plt.show()

# 4. Resposta
print('\n📖 Resposta:')
print('  Sim, em média municípios maiores tendem a ter IDHM mais alto.')
print('  Metrópoles têm a maior média. Porém, o desvio-padrão elevado nas categorias médias')
print('  e grande mostra que existem exceções relevantes: pequenos municípios do interior')
print('  do Sul podem ter IDHM comparável a grandes cidades do Norte/Nordeste.')
print('  Portanto: tamanho é um fator associado, mas não determinante.')

---
## ✅ Gabarito — Exercício 7: Painel Final

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9))
fig.suptitle('Desenvolvimento Humano nos Municípios Brasileiros — Painel de EDA',
             fontsize=14, fontweight='bold')

# --- Gráfico 1: Histograma ---
ax1 = axes[0, 0]
ax1.hist(df['IDHM'].dropna(), bins=35, color='steelblue', edgecolor='white', alpha=0.85)
ax1.axvline(df['IDHM'].mean(), color='red', linestyle='--', linewidth=2, label=f'Média: {df["IDHM"].mean():.3f}')
ax1.set_title('① Distribuição do IDHM', fontsize=11, fontweight='bold')
ax1.set_xlabel('IDHM'); ax1.set_ylabel('Municípios')
ax1.legend(fontsize=9)

# --- Gráfico 2: IDHM médio por Região ---
ax2 = axes[0, 1]
med_reg = df.groupby('REGION')['IDHM'].mean().sort_values()
cores_r = [COR_REGIAO[r] for r in med_reg.index]
ax2.barh(med_reg.index, med_reg.values, color=cores_r, edgecolor='white')
for i, v in enumerate(med_reg.values):
    ax2.text(v + 0.002, i, f'{v:.3f}', va='center', fontsize=9)
ax2.set_title('② IDHM Médio por Região', fontsize=11, fontweight='bold')
ax2.set_xlabel('IDHM Médio')
ax2.set_xlim(0.55, 0.82)

# --- Gráfico 3: Componentes por Região ---
ax3 = axes[1, 0]
comp_reg = df.groupby('REGION')[['IDHM_Renda','IDHM_Longevidade','IDHM_Educacao']].mean()
comp_reg.columns = ['Renda','Longevidade','Educação']
comp_reg.plot(kind='bar', ax=ax3, width=0.7, color=['#e6994c','#56b4e9','#009e73'], edgecolor='white')
ax3.set_title('③ Componentes do IDHM por Região', fontsize=11, fontweight='bold')
ax3.set_ylabel('Valor'); ax3.tick_params(axis='x', rotation=20)
ax3.set_ylim(0.5, 0.82)
ax3.legend(title='Componente', fontsize=8)

# --- Gráfico 4: Scatter GDP x IDHM ---
ax4 = axes[1, 1]
p95 = df['GDP_CAPITA'].quantile(0.95)
df_s = df[df['GDP_CAPITA'] <= p95].dropna(subset=['GDP_CAPITA', 'IDHM'])
if len(df_s) > 600:
    df_s = df_s.sample(600, random_state=42)
for reg, grp in df_s.groupby('REGION'):
    ax4.scatter(grp['GDP_CAPITA'], grp['IDHM'], color=COR_REGIAO[reg], alpha=0.35, s=12)
ax4.set_title('④ PIB per Capita × IDHM', fontsize=11, fontweight='bold')
ax4.set_xlabel('PIB per Capita (R$)'); ax4.set_ylabel('IDHM')
ax4.xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f'R${x/1000:.0f}k'))

plt.tight_layout()
plt.show()
print('✅ Painel gerado com sucesso!')

---
## ✅ Gabarito — Exercício 8: Storytelling

**Narrativa — Modelo de Resposta:**

---

**Insight 1 — Dois Brasis no mesmo mapa**  
A análise revela uma divisão clara: enquanto municípios do Sul e Sudeste têm IDHM médio acima de 0,73, municípios do Nordeste e Norte ficam abaixo de 0,62. Isso não é apenas uma diferença estatística — representa anos de defasagem em expectativa de vida, escolaridade e renda. O boxplot por região (Gráfico 2) mostra que essa desigualdade existe tanto *entre* regiões quanto *dentro* de cada uma delas.

**Insight 2 — Educação é o elo mais fraco**  
O gráfico de componentes do IDHM (Gráfico 3) mostra que Educação é consistentemente o menor dos três componentes em todas as regiões. Já a Longevidade — impulsionada por décadas de investimento no SUS e em vacinação — é o componente mais alto e mais equitativo. Isso indica onde as políticas públicas devem concentrar esforços nos próximos anos.

**Insight 3 — Riqueza e desenvolvimento nem sempre andam juntos**  
O scatter plot (Gráfico 4) mostra correlação positiva entre PIB per capita e IDHM, mas com grande dispersão. Existem municípios com PIB per capita elevado e IDHM mediano — o que pode indicar concentração de renda que não se traduz em bem-estar coletivo. Também há municípios pequenos com IDHM surpreendentemente alto, mostrando que comunidade coesa e serviços públicos eficientes podem compensar a falta de riqueza bruta.

**Recomendação de Política Pública:**  
Os dados sugerem que investir em **educação de qualidade** — especialmente no Norte e Nordeste — tem o maior potencial de elevar o IDHM nacional. Programas de alfabetização de adultos, melhoria da infraestrutura escolar e acesso ao ensino médio em municípios de pequeno porte deveriam ser prioritários, dado que Educação é o gargalo em todas as regiões.

---
*Este é apenas um modelo. Respostas com argumentos diferentes, mas bem fundamentados em dados, também estão corretas.*